In [ ]:
QDRANT_URL="https://7cde0ad2-400d-4b71-8eb9-a8eb851f15fe.us-west-2-0.aws.cloud.qdrant.io"
QDRANT_API_KEY="YOUR_QDRANT_API_KEY"
COLLECTION_NAME="Computer Science and AI engineer books"
MODEL_EMBEDDING="sentence-transformers/all-MiniLM-L6-v2"
GEMINI_API_KEY='YOUR_GEMINI_API_KEY'

In [ ]:
# Cell 1: Install dependencies (skip if already installed)
!pip install -q qdrant-client google-genai
!pip install -q sentence-transformers
!pip install -q fastembed

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 22.9 MB/s eta 0:00:00


In [ ]:
# Cell 2: Connect to Qdrant and inspect the collection config
from qdrant_client import QdrantClient

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

# 1. Basic collection info: vector size, distance metric, named vectors, etc.
collection_info = client.get_collection(collection_name=COLLECTION_NAME)
print("=== COLLECTION INFO ===")
print(collection_info)

=== COLLECTION INFO ===
status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=9167 points_count=9167 segments_count=2 config=CollectionConfig(params=CollectionParams(vectors={'text-dense': VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None)}, shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, sparse_vectors={'text-sparse-new': SparseVectorParams(index=SparseIndexParams(full_scan_threshold=None, on_disk=None, datatype=None), modifier=None)}), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(MODEL_EMBEDDING)
print("Model loaded:", MODEL_EMBEDDING)
print("Max seq length:", model.max_seq_length)
print("Embedding dimension:", model.get_sentence_embedding_dimension())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded: sentence-transformers/all-MiniLM-L6-v2
Max seq length: 256
Embedding dimension: 384


/tmp/ipykernel_4993/1212399791.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


In [ ]:
from qdrant_client import models
from fastembed import SparseTextEmbedding

sparse_model = SparseTextEmbedding(model_name="Qdrant/bm25")

def retrieve(query, sparse_k=5, top_k=5):
    dense_k = sparse_k * 2

    # Dense vector via SentenceTransformer
    dense_vector = model.encode(query).tolist()

    # Sparse vector via FastEmbed BM25
    sparse_embedding = list(sparse_model.embed([query]))[0]
    sparse_vector = models.SparseVector(
        indices=sparse_embedding.indices.tolist(),
        values=sparse_embedding.values.tolist(),
    )

    results = client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[
            models.Prefetch(
                query=dense_vector,
                using="text-dense",
                limit=dense_k,
            ),
            models.Prefetch(
                query=sparse_vector,
                using="text-sparse-new",
                limit=sparse_k,
            ),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=top_k,
        with_payload=True,
    )

    return results.points

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

In [ ]:
# Cell: Print payloads + extracted text chunk (parsing _node_content)
import json

results = retrieve("can you explain to me the binary search", sparse_k=5, top_k=5)

for i, point in enumerate(results):
    print(f"\n--- Result {i+1} (score: {point.score}) ---")

    payload = point.payload
    print("book_name:", payload.get("book_name"))
    print("source:", payload.get("source"))
    print("page_label:", payload.get("page_label"))

    # The text lives inside _node_content, which is a JSON string
    node_content = payload.get("_node_content")
    text_chunk = None
    if node_content:
        node_data = json.loads(node_content)
        text_chunk = node_data.get("text")

    print("Text chunk:\n", text_chunk)


--- Result 1 (score: 0.5) ---
book_name: Grokking Algorithms
source: Grokking Algorithms, Second Edition -- Aditya Y. Bhargava -- ( WeLib.org ).pdf
page_label: 30
Text chunk:
 Chapter 1  I  introduction to algorithms
4
Here’s an example.
Looking for companies 
in a phone book with 
binary search
Now here’s an example of how binary search works. I’m thinking of a 
number between 1 and 100.
You have to try to guess my number in the fewest tries possible. With 
every guess, I’ll tell you if your guess is too low, too high, or correct.
Suppose you start guessing like this: 1, 2, 3, 4, . . . . Here’s how it would go.

--- Result 2 (score: 0.33333334) ---
book_name: Grokking Algorithms
source: Grokking Algorithms, Second Edition -- Aditya Y. Bhargava -- ( WeLib.org ).pdf
page_label: 37
Text chunk:
 Big O notation
11
Algorithm running times grow at different rates
Bob is writing a search algorithm for NASA. His algorithm will kick in 
when a rocket is about to land on the Moon, and it will h

In [ ]:
# Cell: RAG answer function - retrieve context, then ask Gemini to answer only from it
from google import genai

gemini_client = genai.Client(api_key=GEMINI_API_KEY)

def build_context(results):
    context_blocks = []
    for i, point in enumerate(results):
        payload = point.payload
        node_data = json.loads(payload.get("_node_content", "{}"))
        text_chunk = node_data.get("text", "")
        book = payload.get("book_name", "Unknown")
        page = payload.get("page_label", "?")
        context_blocks.append(f"[Source {i+1} | {book}, page {page}]\n{text_chunk}")
    return "\n\n---\n\n".join(context_blocks)


def rag_answer(query, sparse_k=5, top_k=5):
    results = retrieve(query, sparse_k=sparse_k, top_k=top_k)
    context = build_context(results)

    prompt = f"""Answer the question using ONLY the information in the context below.
If the context does not contain enough information to answer, say "I cannot answer this based on the provided documents."
Do not use any outside knowledge. Cite book name at the end of response make the response clean. You only can write the code if the user ask from you to do so you must write it in clean python
and if the resource list the code you need to refactor it in python if it is in another language.

Context:
{context}

Question: {query}

Answer:"""

    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )

    return response.text, results



In [ ]:
# Example usage:
answer, sources = rag_answer("Can you define what is the importance of event loop in async what will happen if I include synchronous request into this event loop")
print(answer)

The event loop is the main FastAPI server thread responsible for orchestrating the processing of requests. It is the core component of every application built on top of `asyncio`, including FastAPI, that implements concurrency. Event loops run asynchronous tasks and callbacks, perform network I/O operations, and run subprocesses. In FastAPI, the event loop is also responsible for orchestrating the asynchronous processing of requests, and running handlers on it via asynchronous programming can be more efficient than running them on a thread pool.

If a synchronous request, especially one with a blocking operation, is run directly on the event loop (main server thread), it will block the event loop. This causes other requests to be blocked until the current request is processed. FastAPI typically handles synchronous blocking operations by running sync handlers in its thread pool to prevent blocking the event loop from executing tasks. Synchronous code can be made asynchronous using a pro

In [ ]:
# Cell: ReAct agent (LangGraph) with RAG tool using langchain-google-genai
!pip install -q langchain-google-genai langgraph langchain-core

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.prebuilt import create_react_agent

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    google_api_key=GEMINI_API_KEY,
    temperature=0,
)

@tool
def rag_search(query: str, top_k: int = 5) -> str:
    """Search a knowledge base of technical books (Statistics, Machine Learning, AI Engineering,
    Search Systems, Computer Science, FastAPI, etc.) using hybrid dense+sparse retrieval.
    Call this with a focused query or sub-query to get relevant document chunks.
    If the user's question is complex, break it into smaller sub-questions and call this
    tool separately for each one (you may call it up to 3 times total).

    Args:
        query: the search query or sub-query to retrieve relevant chunks for
        top_k: number of fused results to return (sparse_k = top_k, dense_k = 2*top_k internally)
    """
    results = retrieve(query, sparse_k=top_k, top_k=top_k)
    return build_context(results)


SYSTEM_PROMPT = """You are a specialized assistant that ONLY answers questions about:
- Mathematics (statistics, probability, linear algebra, calculus, etc.)
- AI Engineering (machine learning, deep learning, RAG, search systems, LLMs, etc.)
- Computer Science (algorithms, data structures, software engineering, etc.)
- Anything directly related to the topics above

If the user asks about anything outside these domains, politely decline and explain you only
handle mathematics, AI engineering, and computer science topics.
Tools : You have access to the following tools:
1- You have a `rag_search` tool connected to a knowledge base of technical books.

Rules:
1. ALWAYS use rag_search to gather supporting context before answering domain questions — never answer from memory alone.
2. If the question is complex, decompose it into sub-questions and call rag_search separately for each.
3. You may call rag_search at most 3 times total. Use those calls wisely — investigate first, then answer.
4. After gathering context, synthesize and summarize it clearly in your own words for the user.
5. If retrieved context is insufficient, say so honestly instead of making things up.
6. Cite the book/page of the information you used whenever possible.
7- if something in the user question repeated in your context you can answer from the history safely
"""

# Cell: Add memory/checkpointer for multi-turn conversation state
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()

agent = create_react_agent(
    model=llm,
    tools=[rag_search],
    prompt=SystemMessage(content=SYSTEM_PROMPT),
    checkpointer=memory,
)

def ask_agent(query, thread_id="default-session"):
    config = {"configurable": {"thread_id": thread_id}, "recursion_limit": 10}
    result = agent.invoke(
        {"messages": [HumanMessage(content=query)]},
        config=config,
    )
    return result["messages"][-1].content



/tmp/ipykernel_4993/2791017209.py:57: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [ ]:
# Example usage:
answer = ask_agent("can you ask explain the svm algorithm")
print(answer)

The Support Vector Machine (SVM) algorithm is a method used for binary classification. Its core idea is to find the best hyperplane that separates data points into two classes. A hyperplane is a boundary that divides a vector space into two parts.

Here's a breakdown of how it works:

*   **Hyperplane:** In a 2D space, a hyperplane is a line; in a 3D space, it's a plane. The SVM aims to find the hyperplane that has the maximum margin between the closest data points of the two classes.
*   **Orthogonal Vector:** The hyperplane is defined by a vector that is orthogonal (perpendicular) to it. This vector indicates the direction of relevance for the features.
*   **Weights:** When an SVM library processes data, it provides weights for each feature. These weights are determined by the orthogonal vector and signify the influence of each feature in separating the classes.
*   **Training:** The algorithm is trained on a dataset where data points are labeled into two classes. For example, in a 

In [ ]:
print(ask_agent('what is the loss function of it'))

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 29.746034168s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT